# Getting Started with the Analysis Layer

This notebook demonstrates how to explore the government data lake and load data for analysis.

## Prerequisites

Make sure the MinIO service is running with data in the `gov-data-lake` bucket.

## 1. Import the Analysis Helpers

In [ ]:
from analysis_helpers import (
    MinioExplorer,
    load_document,
    load_table,
    SQLContext,
    query_sql,
    display_table_preview,
    display_asset_tree,
)

## 2. Create an Explorer and Browse the Data Lake

In [ ]:
# Create an explorer (uses environment variables for connection)
explorer = MinioExplorer()

# List available zones
zones = explorer.list_zones()
print("Available zones:", zones)

In [ ]:
# List agencies in the parsed zone
agencies = explorer.list_agencies("parsed-zone")
print("Agencies:", agencies)

In [ ]:
# If agencies exist, list assets for the first one
if agencies:
    agency = agencies[0]
    assets = explorer.list_assets(agency, "parsed-zone")
    display_asset_tree(assets, title=f"Assets for {agency}")

## 3. Load a Document

In [ ]:
# Load the latest version of a document
# Replace with actual agency/asset names from your data
if agencies and assets:
    doc = explorer.load_document(agency, assets[0].asset)
    
    # Show document metadata
    print("Document title:", doc.get("metadata", {}).get("title", "Unknown"))
    print("Source:", doc.get("_source", {}))

In [ ]:
# List tables in the document
if 'doc' in dir():
    tables = explorer.list_tables(doc)
    print(f"Found {len(tables)} table(s):")
    for t in tables:
        # Use 1-based numbering for display, show actual title or indicate none
        title = t['title'] if not t['title'].startswith('Table ') else '(untitled)'
        print(f"  [{t['index']}] {title} - {t['rows']} rows, {t['columns']} columns")

## 4. Load a Table as DataFrame

In [ ]:
# Load the first table
if 'doc' in dir() and tables:
    df = explorer.load_table(doc, table_index=0)
    display_table_preview(df, title=tables[0]['title'])

## 5. Query Data with SQL

In [ ]:
# Use SQLContext for multiple queries
if 'df' in dir():
    ctx = SQLContext()
    ctx.register("data", df)
    
    # Show registered tables
    print("Registered tables:", ctx.tables())
    
    # Describe the table
    ctx.describe("data")

In [ ]:
# Run a query
if 'ctx' in dir():
    result = ctx.query("SELECT * FROM data LIMIT 5")
    display_table_preview(result, title="Query Result")

In [ ]:
# One-off query with query_sql
if 'df' in dir():
    # Count rows
    result = query_sql({"data": df}, "SELECT COUNT(*) as total_rows FROM data")
    print("Total rows:", result['total_rows'].iloc[0])

## 6. Shorthand Loading

For quick access, use the convenience functions:

In [ ]:
# Load document directly
# doc = load_document("uscis", "quarterly-forms")

# Load table directly as DataFrame
# df = load_table("uscis", "quarterly-forms", table_index=0)

## Next Steps

- See `02_table_extraction.ipynb` for working with table data
- See `03_metadata_curation.ipynb` for using the LLM-powered metadata assistant